# TMDL Spike: Looking for Prep for AI Settings in `getDefinition`

Companion notebook to `docs/research/tmdl-prep-for-ai-spike.md`.

**What this notebook does**

1. Authenticates against Fabric.
2. Calls `POST .../semanticModels/{id}/getDefinition` for a target model.
3. Decodes the returned TMDL files.
4. Searches each file for AI Instructions / Verified Answers hint strings.
5. Records what was found (or not found), with file paths and surrounding snippets.

**What this notebook does not do**

It does not call `updateDefinition`. Writes are out of scope for the spike.

**Prerequisites**

- Run inside a Fabric notebook (or anywhere with a Power BI / Fabric bearer token).
- The target semantic model has at least one AI Instruction and one Verified Answer configured in the Fabric UI before running this notebook. Without configured settings, absence of a hit proves nothing.

## 1. Install

In [ ]:
%pip install fabric-ai-meta

## 2. Configure target model

Replace the placeholders below with your workspace and semantic model identifiers (GUIDs).

In [ ]:
WORKSPACE_ID = "YOUR_WORKSPACE_GUID"
MODEL_ID = "YOUR_SEMANTIC_MODEL_GUID"

## 3. Acquire a Fabric bearer token

Inside a Fabric notebook, `notebookutils.credentials.getToken('pbi')` returns a token scoped for Power BI / Fabric. Outside Fabric, use an `azure.identity` credential (e.g., `InteractiveBrowserCredential` or a service principal) acquired with the `https://api.fabric.microsoft.com/.default` scope.

In [ ]:
try:
    import notebookutils  # type: ignore[import-not-found]
    token = notebookutils.credentials.getToken("pbi")
    credential = token  # TMDLClient accepts a raw bearer string.
    print("Acquired Fabric notebook token.")
except ImportError:
    from azure.identity import InteractiveBrowserCredential
    credential = InteractiveBrowserCredential()
    print("Falling back to InteractiveBrowserCredential.")

## 4. Fetch the TMDL definition

Call `getDefinition` and inspect the file list.

In [ ]:
from fabric_ai_meta.writeback.tmdl_client import TMDLClient

client = TMDLClient(credential, WORKSPACE_ID)
definition = client.get_definition(MODEL_ID)

paths = [p["path"] for p in definition["definition"]["parts"]]
print(f"Definition contains {len(paths)} TMDL files:")
for path in paths:
    print(f"  - {path}")

## 5. Search for Prep for AI hints

`find_prep_for_ai_settings` scans every decoded TMDL file for known hint strings (e.g., `__PBI_AIInstructions`, `VerifiedAnswers`). Matches include the file path and a snippet of surrounding text so the storage shape (annotation vs extended property vs first-class) can be classified.

In [ ]:
result = client.find_prep_for_ai_settings(definition)

if result is None:
    print("No hint strings matched. Either the settings live outside the TMDL payload,")
    print("or the hint list needs to be expanded.")
else:
    print(f"Found {len(result['matches'])} matches:")
    for match in result["matches"]:
        print(f"\n--- {match['path']}  (hint: {match['hint']}) ---")
        print(match["snippet"])

## 6. Inspect the model.tmdl file directly

Even when hint strings do not match, the `model.tmdl` file usually contains the model-level annotations. Print the first few KB to see the annotation shape Microsoft is currently using.

In [ ]:
import base64

for part in definition["definition"]["parts"]:
    if part["path"].endswith("model.tmdl"):
        text = base64.b64decode(part["payload"]).decode("utf-8", errors="replace")
        print(text[:4000])
        break

## 7. Record findings

Update `docs/research/tmdl-prep-for-ai-spike.md` with the results of cells 5 and 6:

- **Q1 (AI Instructions in TMDL?):** yes / no, with file path and annotation key.
- **Q2 (Verified Answers in TMDL?):** yes / no, with file path and annotation key.
- **Q3 (Storage shape?):** annotation, extended property, or first-class TMDL property.

Once Q1 and Q2 resolve to yes, the door is open for Sprint 7 to build a TMDL writer with the round-trip preservation constraints listed in the research doc.